Import Library

In [20]:
# ============================================================
# IMPORT LIBRARY
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd

from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score

Load Dataset

In [21]:
# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(
    "../data/processed/cases.csv"
)

print("Jumlah Kasus :", len(df))

df.head()

Jumlah Kasus : 100


,case_id,file_name,no_perkara,tanggal,pasal,ringkasan_fakta,amar_putusan,jumlah_kata,jumlah_karakter,text_full
0,1,case_001.txt,1013 k pid.sus 2026,2025-08-07,"pasal 2, pasal 76, pasal 88",berdasarkan fakta hukum yang relevan secara yu...,mengadili telah dilaksanakan menurut ketentu...,1351,9333,gnomor 1013 k pid.sus 2026 memeriksa perkara t...
1,2,case_002.txt,1013 k pid.sus 2026,2025-08-07,"pasal 2, pasal 76, pasal 88",berdasarkan fakta hukum yang relevan secara yu...,mengadili telah dilaksanakan menurut ketentu...,1351,9333,gnomor 1013 k pid.sus 2026 memeriksa perkara t...
2,3,case_003.txt,Tidak Ditemukan,2026-04-06,"pasal 1, pasal 17, pasal 204, pasal 250, pasal...","berdasarkan fakta hukum tersebut di atas, terd...",mengadili perkara pidana dengan acara pemeri...,21672,138199,nomor 103 pid.sus 2025 pn pts pengadilan n...
3,4,case_004.txt,1092 k pid 2024,2024-03-06,"pasal 12, pasal 2, pasal 30",gnomor 1092 k pid 2024 memeriksa perkara tinda...,dinyatakan ditolak dengan perbaikan e menimnba...,1230,8511,gnomor 1092 k pid 2024 memeriksa perkara tinda...
4,5,case_005.txt,1153 pk pid.sus 2024,2024-08-07,"pasal 2, pasal 263, pasal 266, pasal 296, pasa...",gnomor 1153 pk pid.sus 2024 memeriksa perkara ...,dinyatakan ditolak dan putusan yang dimohonkan...,882,6185,gnomor 1153 pk pid.sus 2024 memeriksa perkara ...


Build Amar Singkat

In [22]:
# ============================================================
# AMAR PUTUSAN SINGKAT
# ============================================================

def simplify_amar(text):

    text = str(text).lower()

    if "ditolak dengan perbaikan" in text:
        return "ditolak dengan perbaikan"

    elif "dinyatakan ditolak" in text:
        return "dinyatakan ditolak"

    elif "menolak permohonan" in text:
        return "menolak permohonan"

    elif "mengabulkan permohonan" in text:
        return "mengabulkan permohonan"

    elif "mengadili" in text:
        return "mengadili"

    return "lainnya"


df["amar_singkat"] = df["amar_putusan"].apply(
    simplify_amar
)

df[
    [
        "no_perkara",
        "amar_singkat"
    ]
].head()

,no_perkara,amar_singkat
0,1013 k pid.sus 2026,mengadili
1,1013 k pid.sus 2026,mengadili
2,Tidak Ditemukan,mengadili
3,1092 k pid 2024,ditolak dengan perbaikan
4,1153 pk pid.sus 2024,dinyatakan ditolak


Build Retrieval Text

In [23]:
# ============================================================
# RETRIEVAL TEXT
# ============================================================

df["retrieval_text"] = (

    df["pasal"].astype(str)

    + " "

    + df["ringkasan_fakta"].astype(str)

    + " "

    + df["text_full"].astype(str).str[:5000]

)

print(
    df["retrieval_text"].iloc[0][:300]
)

pasal 2, pasal 76, pasal 88 berdasarkan fakta hukum yang relevan secara yuridis yang h terungkap di muka sidaneg, sekitar bulan mei 2024 terdakwa bekerja a sebagai manajer di jrawa jawa club yang beralamat di jalan melawai vi,   s ruko blok m rt 03 rw 01, kelurahan melawai, kecamatan kebayoran m bar


Split Data

In [24]:
# ============================================================
# TRAIN TEST SPLIT
# ============================================================

train_df, test_df = train_test_split(

    df,

    test_size=0.20,

    random_state=42

)

print("Train :", len(train_df))
print("Test  :", len(test_df))

Train : 80
Test  : 20


Load TF-IDF Vectorizer

In [25]:
# ============================================================
# LOAD TF-IDF MODEL
# ============================================================

vectorizer = joblib.load(
    "../models/tfidf_vectorizer.pkl"
)

print(
    "TF-IDF Vectorizer berhasil dimuat"
)

TF-IDF Vectorizer berhasil dimuat


Bangun Matrix Train

In [26]:
# ============================================================
# TRANSFORM TRAIN DATA
# ============================================================

X_train = vectorizer.transform(

    train_df["retrieval_text"]

)

print(X_train.shape)

(80, 4971)


Fungsi Retrieval

In [27]:
# ============================================================
# RETRIEVE FUNCTION
# ============================================================

def retrieve(

    query,

    k=5

):

    query_vector = vectorizer.transform(
        [query]
    )

    similarity_scores = cosine_similarity(

        query_vector,

        X_train

    )[0]

    top_idx = np.argsort(

        similarity_scores

    )[::-1][:k]

    results = train_df.iloc[top_idx][

        [

            "case_id",

            "no_perkara",

            "tanggal",

            "pasal",

            "ringkasan_fakta",

            "amar_singkat"

        ]

    ].copy()

    results["similarity"] = (

        similarity_scores[top_idx]

    )

    return results

Uji Retrieval

In [28]:
# ============================================================
# TEST RETRIEVAL
# ============================================================

query_case = test_df.iloc[0]

retrieve(

    query_case["retrieval_text"],

    k=5

)

,case_id,no_perkara,tanggal,pasal,ringkasan_fakta,amar_singkat,similarity
82,83,598 k pid.sus 2026,2025-07-22,"pasal 55, pasal 4, pasal 55, pasal 68, pasal...",gnomor 598 k pid.sus 2026 memeriksa perkara ti...,mengadili,1.000000
38,39,2052 k pid.sus 2026,2025-07-22,"pasal 10, pasal 20, pasal 3, pasal 361, pasal ...",gnomor 2052 k pid.sus 2026 memeriksa perkara t...,mengadili,0.631367
9,10,12069 k pid.sus 2025,2025-09-25,"pasal 11, pasal 2, pasal 55, pasal 69, pasal 81",gnomor 12069 k pid.sus 2025 memeriksa perkara ...,mengadili,0.413295
35,36,2014 k pid.sus 2026,2025-09-24,"pasal 2, pasal 2, pasal 20, pasal 55, pasal ...",fakta hukum yang relevan secara yuridis dengan...,ditolak dengan perbaikan,0.390901
36,37,2014 k pid.sus 2026,2025-09-24,"pasal 2, pasal 2, pasal 20, pasal 55, pasal ...",fakta hukum yang relevan secara yuridis dengan...,ditolak dengan perbaikan,0.390901


Case Solutions

In [29]:
# ============================================================
# CASE SOLUTIONS
# ============================================================

case_solutions = dict(

    zip(

        train_df["case_id"],

        train_df["amar_singkat"]

    )

)

print(
    "Jumlah Solusi :",
    len(case_solutions)
)

Jumlah Solusi : 80


Majority Vote

In [30]:
# ============================================================
# MAJORITY VOTE
# ============================================================

def majority_vote(
    solutions
):

    votes = Counter(
        solutions
    )

    return votes.most_common(1)[0][0]

Weighted Similarity

In [31]:
# ============================================================
# WEIGHTED VOTE
# ============================================================

def weighted_vote(

    solutions,

    similarities

):

    scores = {}

    for solution, similarity in zip(

        solutions,

        similarities

    ):

        scores[solution] = (

            scores.get(solution, 0)

            + similarity

        )

    return max(

        scores,

        key=scores.get

    )

Predict Outcome

In [32]:
# ============================================================
# PREDICT OUTCOME
# ============================================================

def predict_outcome(
    query
):

    top_k = retrieve(
        query,
        k=5
    )

    solutions = list(

        top_k[
            "amar_singkat"
        ]

    )

    similarities = list(

        top_k[
            "similarity"
        ]

    )

    predicted_solution = weighted_vote(

        solutions,

        similarities

    )

    return predicted_solution

Demo 1 Kasus

In [33]:
# ============================================================
# DEMO SATU KASUS
# ============================================================

query_case = test_df.iloc[0]

print("NO PERKARA")
print(query_case["no_perkara"])

print("\nPUTUSAN ASLI")
print(query_case["amar_singkat"])

print("\nHASIL PREDIKSI")

prediction = predict_outcome(

    query_case["retrieval_text"]

)

print(prediction)

NO PERKARA
598 k pid.sus 2026

PUTUSAN ASLI
mengadili

HASIL PREDIKSI
mengadili


Demo 5 Kasus

In [34]:
# ============================================================
# DEMO 5 KASUS
# ============================================================

demo_results = []

for i in range(5):

    query_case = test_df.iloc[i]

    prediction = predict_outcome(

        query_case["retrieval_text"]

    )

    demo_results.append({

        "case_id":
        query_case["case_id"],

        "actual":
        query_case["amar_singkat"],

        "predicted":
        prediction

    })

demo_df = pd.DataFrame(
    demo_results
)

demo_df

,case_id,actual,predicted
0,84,mengadili,mengadili
1,54,ditolak dengan perbaikan,ditolak dengan perbaikan
2,71,mengadili,mengadili
3,46,ditolak dengan perbaikan,ditolak dengan perbaikan
4,45,menolak permohonan,dinyatakan ditolak


Evaluasi Demo

In [35]:
# ============================================================
# EVALUASI DEMO
# ============================================================

demo_accuracy = accuracy_score(

    demo_df["actual"],

    demo_df["predicted"]

)

print(
    f"Akurasi Demo : {demo_accuracy:.2%}"
)

Akurasi Demo : 80.00%


Evaluasi Test Set

In [36]:
# ============================================================
# EVALUASI TEST SET
# ============================================================

actual_all = []

predicted_all = []

for _, row in test_df.iterrows():

    prediction = predict_outcome(

        row["retrieval_text"]

    )

    actual_all.append(
        row["amar_singkat"]
    )

    predicted_all.append(
        prediction
    )

test_accuracy = accuracy_score(

    actual_all,

    predicted_all

)

print(
    f"Akurasi Test Set : {test_accuracy:.2%}"
)

Akurasi Test Set : 75.00%


Generate Predictions

In [37]:
# ============================================================
# GENERATE PREDICTIONS
# ============================================================

predictions = []

for _, row in test_df.iterrows():

    top_cases = retrieve(

        row["retrieval_text"],

        k=5
        

    )

    prediction = predict_outcome(

        row["retrieval_text"]

    )

    predictions.append({

        "query_id":
        row["case_id"],

        "predicted_solution":
        prediction,

        "top_5_case_ids":
        ",".join(

            map(

                str,

                top_cases[
                    "case_id"
                ].tolist()

            )

        )

    })

pred_df = pd.DataFrame(
    predictions
)

pred_df.head()

,query_id,predicted_solution,top_5_case_ids
0,84,mengadili,"83,39,10,36,37"
1,54,ditolak dengan perbaikan,"29,30,27,26,37"
2,71,mengadili,"25,15,94,53,68"
3,46,ditolak dengan perbaikan,"47,70,8,10,21"
4,45,dinyatakan ditolak,"10,67,48,49,62"


SAVE PREDICTIONS

In [38]:
# ============================================================
# SAVE PREDICTIONS
# ============================================================

os.makedirs(
    "../data/results",
    exist_ok=True
)

pred_df.to_csv(

    "../data/results/predictions.csv",

    index=False

)

print(
    "predictions.csv berhasil dibuat"
)

predictions.csv berhasil dibuat
